# Install dependencies and environment setup

In [1]:
%pip -qqq install miditoolkit pandarallel

In [2]:
from pandarallel import pandarallel
pandarallel.initialize()

INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


# Import Dataset from Kaggle

In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

directory = kagglehub.dataset_download(
  "blanderbuss/midi-classic-music"
)

Using Colab cache for faster access to the 'midi-classic-music' dataset.


# Restructure and filter midi_files

Filter for Bach, Beethoven, Chopin, and Mozart midi files only

In [4]:
import pandas as pd
import glob

def read_midi_files(composers):
  midi_files = []
  for composer in composers:
    midi_filenames = glob.glob(f'{directory}/midiclassics/{composer}/*.mid')
    midi_filenames += glob.glob(f'{directory}/midiclassics/{composer}/**/*.mid')
    print(f'Found {len(midi_filenames)} .mid files for {composer}')
    midi_files += [{
        'composer': composer,
        'filename': filename
    } for filename in midi_filenames]

  df = pd.DataFrame(midi_files)
  return df

df = read_midi_files(['Bach', 'Beethoven', 'Chopin', 'Mozart'])
df

Found 876 .mid files for Bach
Found 212 .mid files for Beethoven
Found 136 .mid files for Chopin
Found 219 .mid files for Mozart


,composer,filename
0,Bach,/kaggle/input/midi-classic-music/midiclassics/...
1,Bach,/kaggle/input/midi-classic-music/midiclassics/...
2,Bach,/kaggle/input/midi-classic-music/midiclassics/...
3,Bach,/kaggle/input/midi-classic-music/midiclassics/...
4,Bach,/kaggle/input/midi-classic-music/midiclassics/...
...,...,...
1438,Mozart,/kaggle/input/midi-classic-music/midiclassics/...
1439,Mozart,/kaggle/input/midi-classic-music/midiclassics/...
1440,Mozart,/kaggle/input/midi-classic-music/midiclassics/...
1441,Mozart,/kaggle/input/midi-classic-music/midiclassics/...


# Data pre-processing and feature extraction

The `Composer_Dataset` directory contains 3 subdirectories: `dev`, `test`, and `train`. Each subdirectory contains additional subdirectories for each composer, which contain .mid (MIDI) files. The files must be pre-processed to extract relevant features for our deep learning model. The following features may be extracted from the MIDI files:

- Key signature
- Time signature
- Tempo
- Sequence of notes (pitch, duration, velocity)
- Instrumentation
- etc.

In [5]:
from miditoolkit import MidiFile

def read_midi(filename):
    try:
        midi_obj = MidiFile(filename)
        extracted_data = {
            'ticks_per_beat': midi_obj.ticks_per_beat,
            'max_tick': midi_obj.max_tick,
            'tempo_changes_count': len(midi_obj.tempo_changes),
            'time_signature_changes_count': len(midi_obj.time_signature_changes),
            'key_signature_changes_count': len(midi_obj.key_signature_changes),
            'num_instruments': midi_obj.num_instruments,
            'instrument_names': [inst.name for inst in midi_obj.instruments],
            'instruments_data': {
                inst.name: {
                    'is_drum': inst.is_drum,
                    'program': inst.program,
                    'notes': [{
                        'pitch': note.pitch,
                        'start': note.start,
                        'end': note.end,
                        'duration': note.duration,
                        'velocity': note.velocity
                    } for note in inst.notes]
                } for inst in midi_obj.instruments
            }
        }
        return extracted_data
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return None

df['midi_data'] = df['filename'].parallel_apply(read_midi)

Error reading /kaggle/input/midi-classic-music/midiclassics/Beethoven/Anhang 14-3.mid: Could not decode key with 3 flats and mode 255


In [6]:
# Remove midi files that cannot be parsed
before_len = len(df)
df = df.dropna()
after_len = len(df)
print(f'Removed {before_len - after_len} midi file(s) that cannot be parsed')

Removed 1 midi file(s) that cannot be parsed


# Model Building

Define model architecture utilizing CNN and LSTMs. Reference existing research papers and articles for inspiration.

# Model Training

Train the model using the processed dataset.

# Model Evaluation

Evaluate model performance using the following performance metrics:

- Accuracy
- Precision
- Recall
- F1-score
- AUC-ROC

Use the following visualization techniques to analyze the model's performance:
- Plot training and validation loss curves
- Plot confusion matrix

# Model Optimization

Optimize the model using hyperparameter tuning.